# Flash Point Modeling using DeepONets

This notebook evaluates the use of Deep Operator Networks (DeepONets) for flash point (FP) prediction and compares their performance with a standard feedforward neural network (FNN) baseline. To keep the comparison as fair as possible, both models are trained on the same experimental dataset using similar architectural settings, while still respecting the structural differences between a conventional FNN and a DeepONet.

The notebook is organized into the following stages:

- **Dependencies:** import the required libraries and define the execution setup.
- **Project Utilities:** import the FNN, DeepONet, training, metric, plotting, and Optuna helpers from `src/`.
- **Data Loading:** read the experimental dataset and filter the subset used for modeling.
- **Hyperparameter Optimization:** run the Optuna objective functions for both FNN and DeepONet models, aiming to find the optimal learning rate.
- **Models Training:** prepare the input features, apply 5-fold cross-validation, train both models, and track fold-level performance.
- **Models Benchmarking:** aggregate the results across folds to compare the predictive performance of both approaches.

## Dependencies

In [1]:
# Imports
import copy
import optuna
import pandas as pd
import torch

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# Initial setup
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 10000
EARLY_STOPPING_PATIENCE = 1000
EARLY_STOPPING_MIN_DELTA = 1e-4
DATA_PATH = Path("../data/public")


c:\Users\mauri\anaconda3\envs\sciml_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Project Utilities

In [2]:
from biofuels_sciml.deeponets import (
    DeepONet,
    FNN,
    compute_metrics,
    fit_transform_except_x1,
    objective_deeponet,
    objective_fnn,
    plot_fold_results,
    train_deeponet_fold,
    train_fnn_fold,
    transform_except_x1,
)

from biofuels_sciml.evaluation import evaluate_deeponet_binary_system_predictions


## Data Loading

In [ ]:
# Reading the data into a DataFrame
data = pd.read_csv(DATA_PATH / "raw" / "fp_experimental_data.csv")

display(data)

In [ ]:
# Select Method 0 (ASTM D6450) rows and keep relevant columns
filtered_data = data.loc[
    data["Method"] == 0, ["x_1", "MM", "lnPvap", "FP"]
].reset_index(drop=True)

display(filtered_data)

In [ ]:
# Define features for each model
fnn_features = filtered_data[["MM", "lnPvap"]].values
branch_features = filtered_data[["x_1", "MM", "lnPvap"]].values
trunk_features = filtered_data[["x_1"]].values
fp_values = filtered_data["FP"].values

## Hyperparameter Optimization

In [ ]:
study_fnn = optuna.create_study(direction="minimize")
study_fnn.optimize(
    lambda trial: objective_fnn(
        trial,
        fnn_features,
        fp_values,
        DEVICE,
        epochs=EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
    ),
    n_trials=25,
)

study_deeponet = optuna.create_study(direction="minimize")
study_deeponet.optimize(
    lambda trial: objective_deeponet(
        trial,
        branch_features,
        trunk_features,
        fp_values,
        DEVICE,
        epochs=EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
    ),
    n_trials=25,
)

print("Best FNN:")
print(study_fnn.best_params)
print(study_fnn.best_value)

print("Best DeepONet:")
print(study_deeponet.best_params)
print(study_deeponet.best_value)


## Model Training

In [ ]:
# KFold cross-validation
folds = 5
kf = KFold(n_splits=folds, shuffle=True, random_state=42)

fnn_fold_metrics = []
deeponet_fold_metrics = []
deeponet_fold_artifacts = []
fold_predictions = []

fnn_best_params = study_fnn.best_params
deeponet_best_params = study_deeponet.best_params
fnn_model_params = {
    key: fnn_best_params[key] for key in ("hidden_width", "n_layers", "activation")
}
deeponet_model_params = {
    key: deeponet_best_params[key]
    for key in ("hidden_width", "n_layers", "activation")
}

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(fnn_features), start=1):
    fnn_scaler = StandardScaler()
    branch_scaler = StandardScaler()

    fnn_train = fnn_scaler.fit_transform(fnn_features[train_idx])
    fnn_val = fnn_scaler.transform(fnn_features[val_idx])
    branch_train = fit_transform_except_x1(branch_scaler, branch_features[train_idx])
    branch_val = transform_except_x1(branch_scaler, branch_features[val_idx])
    trunk_train = trunk_features[train_idx]
    trunk_val = trunk_features[val_idx]

    fnn_train_tensor = torch.tensor(fnn_train, dtype=torch.float32).to(DEVICE)
    fnn_val_tensor = torch.tensor(fnn_val, dtype=torch.float32).to(DEVICE)
    branch_train_tensor = torch.tensor(branch_train, dtype=torch.float32).to(DEVICE)
    branch_val_tensor = torch.tensor(branch_val, dtype=torch.float32).to(DEVICE)
    trunk_train_tensor = torch.tensor(trunk_train, dtype=torch.float32).to(DEVICE)
    trunk_val_tensor = torch.tensor(trunk_val, dtype=torch.float32).to(DEVICE)
    y_train_tensor = (
        torch.tensor(fp_values[train_idx], dtype=torch.float32).unsqueeze(1).to(DEVICE)
    )
    y_val_tensor = (
        torch.tensor(fp_values[val_idx], dtype=torch.float32).unsqueeze(1).to(DEVICE)
    )

    fnn_model = FNN(
        input_dim=fnn_train_tensor.shape[1],
        output_dim=1,
        **fnn_model_params,
    ).to(DEVICE)
    fnn_train_losses, fnn_val_losses, fnn_val_predictions = train_fnn_fold(
        fnn_model,
        fnn_train_tensor,
        y_train_tensor,
        fnn_val_tensor,
        y_val_tensor,
        lr=fnn_best_params["lr"],
        epochs=EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
    )
    fnn_metrics = compute_metrics(fnn_val_predictions, y_val_tensor)
    fnn_metrics["fold"] = fold_idx
    fnn_fold_metrics.append(fnn_metrics)
    plot_fold_results(
        "FNN",
        fold_idx,
        fnn_train_losses,
        fnn_val_losses,
        y_val_tensor,
        fnn_val_predictions,
    )

    deeponet_model = DeepONet(
        branch_dim=branch_train_tensor.shape[1],
        trunk_dim=trunk_train_tensor.shape[1],
        output_dim=1,
        **deeponet_model_params,
    ).to(DEVICE)
    deeponet_train_losses, deeponet_val_losses, deeponet_val_predictions = (
        train_deeponet_fold(
            deeponet_model,
            branch_train_tensor,
            trunk_train_tensor,
            y_train_tensor,
            branch_val_tensor,
            trunk_val_tensor,
            y_val_tensor,
            lr=deeponet_best_params["lr"],
            epochs=EPOCHS,
            patience=EARLY_STOPPING_PATIENCE,
            min_delta=EARLY_STOPPING_MIN_DELTA,
        )
    )
    deeponet_metrics = compute_metrics(deeponet_val_predictions, y_val_tensor)
    deeponet_metrics["fold"] = fold_idx
    deeponet_fold_metrics.append(deeponet_metrics)
    deeponet_fold_artifacts.append(
        {
            "fold": fold_idx,
            "model_state_dict": copy.deepcopy(deeponet_model.state_dict()),
            "branch_scaler": branch_scaler,
            "model_params": deeponet_model_params.copy(),
            "lr": deeponet_best_params["lr"],
            "rmse (K)": deeponet_metrics["rmse (K)"],
        }
    )
    plot_fold_results(
        "DeepONet",
        fold_idx,
        deeponet_train_losses,
        deeponet_val_losses,
        y_val_tensor,
        deeponet_val_predictions,
    )

    fold_predictions.append(
        {
            "fold": fold_idx,
            "y_true": y_val_tensor.detach().cpu().numpy().ravel(),
            "fnn_predictions": fnn_val_predictions.detach().cpu().numpy().ravel(),
            "deeponet_predictions": deeponet_val_predictions.detach()
            .cpu()
            .numpy()
            .ravel(),
        }
    )


## Models Benchmarking

In [ ]:
# Feedforward metrics
fnn_metrics_df = pd.DataFrame(fnn_fold_metrics)
mean_fnn_metrics = fnn_metrics_df.drop(columns=["fold"]).mean()

display(fnn_metrics_df)
print(mean_fnn_metrics)

In [ ]:
# DeepONet metrics
deeponet_metrics_df = pd.DataFrame(deeponet_fold_metrics)
mean_deeponet_metrics = deeponet_metrics_df.drop(columns=["fold"]).mean()

display(deeponet_metrics_df)
print(mean_deeponet_metrics)

### 1-Butanol + FAEEs Evaluation

Reading the data into a `pandas.DataFrame`:

In [ ]:
but_faee_data = pd.read_csv(DATA_PATH / "processed" / "butanol_faee_data.csv")

display(but_faee_data)

Run the reusable binary-system evaluation flow:

In [ ]:
but_faee_evaluation = evaluate_deeponet_binary_system_predictions(
    data=but_faee_data,
    fold_artifacts=deeponet_fold_artifacts,
    branch_dim=branch_features.shape[1],
    trunk_dim=trunk_features.shape[1],
    device=DEVICE,
    branch_columns=["x_1", "MM", "lnPvap"],
    trunk_columns=["x_1"],
    reference_substance="Butanol",
    model_name="DeepONet",
    title="DeepONet Predicted FP by 1-Butanol + FAEE System",
)

but_faee_predictions = but_faee_evaluation.predictions
best_deeponet_artifact = but_faee_evaluation.artifact
best_deeponet_model = but_faee_evaluation.model

print(f"Using DeepONet fold {best_deeponet_artifact['fold']} for inference")
display(but_faee_predictions)
